In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.svm import SVR
from sklearn.feature_selection import RFE, f_regression, mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# -----------------------------------------------------------------------------
# 1. RUTAS Y CONFIGURACIÓN
# -----------------------------------------------------------------------------
PATH_RAW = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\5_svm\2_datos\1_raw\2_meteo_epi_2021-2026_1_rezagos.xlsx"
PATH_PROCESSED = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\5_svm\2_datos\2_procesados"
PATH_RESULTS = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\5_svm\3_resultados"

os.makedirs(PATH_PROCESSED, exist_ok=True)
os.makedirs(PATH_RESULTS, exist_ok=True)

# -----------------------------------------------------------------------------
# 2. CARGA DE DATOS E INGENIERÍA DE ATRIBUTOS
# -----------------------------------------------------------------------------
df = pd.read_excel(PATH_RAW)

if 'fecha' in df.columns:
    df['fecha'] = pd.to_datetime(df['fecha'])
    df = df.sort_values('fecha').reset_index(drop=True)

# Transformaciones cíclicas de la temporalidad (conservadas para contexto)
if 'semana_epi' in df.columns:
    df['semana_sin'] = np.sin(2 * np.pi * df['semana_epi'] / 52.0)
    df['semana_cos'] = np.cos(2 * np.pi * df['semana_epi'] / 52.0)

# Variables meteorológicas (NO se usarán como predictores, solo se mantienen para referencia)
if 'temp' in df.columns:
    df['temp_roll_mean_4'] = df['temp'].rolling(window=4, min_periods=1).mean()
if 'prec' in df.columns:
    df['prec_roll_sum_4'] = df['prec'].rolling(window=4, min_periods=1).sum()

# DIFERENCIACIÓN DE LA VARIABLE OBJETIVO
df['casos_dengue_diff'] = df['casos_dengue'].diff()
df['casos_dengue_prev'] = df['casos_dengue'].shift(1)

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

df.to_excel(os.path.join(PATH_PROCESSED, "dataset_feature_engineering.xlsx"), index=False)

# -----------------------------------------------------------------------------
# 3. PREPARACIÓN Y ESCALAMIENTO COMBINADO DE DATOS
# -----------------------------------------------------------------------------
target_col = 'casos_dengue_diff'

# DEFINIR SOLO PREDICTORES AUTOREGRESIVOS (rezagos de casos_dengue)
# Conservar todos los rezagos disponibles para mejor capacidad predictiva
autoregressive_cols = [col for col in df.columns if 'casos_dengue_lag_' in col and col != 'casos_dengue_lag_0']

# Si no hay rezagos disponibles, crear algunos básicos
if len(autoregressive_cols) == 0:
    print("No se encontraron rezagos de casos_dengue. Creando rezagos automáticos...")
    for lag in range(1, 13):
        df[f'casos_dengue_lag_{lag}'] = df['casos_dengue'].shift(lag)
    autoregressive_cols = [f'casos_dengue_lag_{i}' for i in range(1, 13)]
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

# Excluir cualquier columna no autoregresiva
feature_cols = autoregressive_cols

X_raw = df[feature_cols].copy()
y_diff = df[target_col].copy()

# 1. Target: RobustScaler para amortiguar picos y conservar la variabilidad
scaler_y = RobustScaler()
y_scaled = scaler_y.fit_transform(y_diff.values.reshape(-1, 1)).ravel()

# 2. Predictores: StandardScaler para homogeneizar distancias en el Kernel RBF
scaler_X = StandardScaler()
X_scaled_array = scaler_X.fit_transform(X_raw)
X_scaled = pd.DataFrame(X_scaled_array, columns=feature_cols)

# Guardar dataset escalado
df_scaled = X_scaled.copy()
df_scaled[target_col + '_scaled'] = y_scaled
if 'fecha' in df.columns:
    df_scaled['fecha'] = df['fecha']
df_scaled.to_excel(os.path.join(PATH_PROCESSED, "dataset_escalado_autoregresivo.xlsx"), index=False)

# -----------------------------------------------------------------------------
# 4. REDUCCIÓN DIMENSIONAL (SELECCIÓN DE REZAGOS MÁS RELEVANTES)
# -----------------------------------------------------------------------------
def evaluate_feature_selection(X, y):
    methods_results = {}
    
    mi_scores = mutual_info_regression(X, y, random_state=42)
    methods_results['Mutual_Info'] = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
    
    f_scores, _ = f_regression(X, y)
    methods_results['F_Regression'] = pd.Series(np.nan_to_num(f_scores), index=X.columns).sort_values(ascending=False)
    
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X, y)
    methods_results['Random_Forest'] = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

    return methods_results

dim_red_results = evaluate_feature_selection(X_scaled, y_scaled)

# Seleccionar top rezagos más relevantes para minimizar MAE
top_k = 8  # Usar menos features para evitar overfitting y mejorar MAE
selected_features = dim_red_results['Random_Forest'].head(top_k).index.tolist()

df_importance = pd.DataFrame({
    'Atributo': selected_features,
    'Importancia_RF': dim_red_results['Random_Forest'].head(top_k).values,
    'Criterio': 'Random Forest Feature Importance (MDI)'
})
df_importance.to_excel(os.path.join(PATH_RESULTS, "atributos_autoregresivos_seleccionados.xlsx"), index=False)

X_reduced = X_scaled[selected_features].copy()
df_reduced = X_reduced.copy()
df_reduced[target_col + '_scaled'] = y_scaled
if 'fecha' in df.columns:
    df_reduced['fecha'] = df['fecha']
df_reduced.to_excel(os.path.join(PATH_PROCESSED, "dataset_reduccion_dimensional_autoregresivo.xlsx"), index=False)

# -----------------------------------------------------------------------------
# 5. ENTRENAMIENTO, EVALUACIÓN Y SELECCIÓN DEL MEJOR SPLIT
# -----------------------------------------------------------------------------
splits = {
    '80-20': 0.20,
    '90-10': 0.10,
    '95-5': 0.05
}

best_overall_mae = float('inf')
best_split_name = None
best_model = None
best_split_data = None

results_summary = []
y_real_cases = df['casos_dengue'].values
y_prev_cases = df['casos_dengue_prev'].values
peak_threshold = np.percentile(y_real_cases, 75)

for split_name, test_size in splits.items():
    split_idx = int(len(X_reduced) * (1 - test_size))
    
    X_train, X_test = X_reduced.iloc[:split_idx], X_reduced.iloc[split_idx:]
    y_train_sc, y_test_sc = y_scaled[:split_idx], y_scaled[split_idx:]
    
    # Malla de hiperparámetros optimizada para MAE < 4 con datos autoregresivos
    param_grid = {
        'C': [0.5, 1, 2, 5, 8, 12, 20],
        'gamma': [0.0005, 0.001, 0.002, 0.005, 0.01, 0.02],
        'epsilon': [0.001, 0.002, 0.005, 0.01, 0.02, 0.05],
        'kernel': ['rbf']
    }
    
    tscv = TimeSeriesSplit(n_splits=5)
    grid = GridSearchCV(SVR(), param_grid, cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
    grid.fit(X_train, y_train_sc)
    
    model = grid.best_estimator_

    # Predicciones
    pred_train_sc = model.predict(X_train)
    pred_test_sc = model.predict(X_test)

    # Inversa de transformación
    pred_train_diff = scaler_y.inverse_transform(pred_train_sc.reshape(-1, 1)).ravel()
    pred_test_diff = scaler_y.inverse_transform(pred_test_sc.reshape(-1, 1)).ravel()

    # Reconstrucción de la serie original
    pred_train_real = np.maximum(0, y_prev_cases[:split_idx] + pred_train_diff)
    pred_test_real = np.maximum(0, y_prev_cases[split_idx:] + pred_test_diff)

    y_train_real = y_real_cases[:split_idx]
    y_test_real = y_real_cases[split_idx:]

    # Métricas Globales
    mae_train = mean_absolute_error(y_train_real, pred_train_real)
    mae_test = mean_absolute_error(y_test_real, pred_test_real)
    mae_avg = (mae_train + mae_test) / 2.0

    rmse_train = np.sqrt(mean_squared_error(y_train_real, pred_train_real))
    rmse_test = np.sqrt(mean_squared_error(y_test_real, pred_test_real))

    r2_train = r2_score(y_train_real, pred_train_real)
    r2_test = r2_score(y_test_real, pred_test_real)

    # Métricas de Picos
    peak_mask_train = y_train_real >= peak_threshold
    peak_mask_test = y_test_real >= peak_threshold
    
    mae_picos_train = mean_absolute_error(y_train_real[peak_mask_train], pred_train_real[peak_mask_train]) if np.sum(peak_mask_train) > 0 else 0
    mae_picos_test = mean_absolute_error(y_test_real[peak_mask_test], pred_test_real[peak_mask_test]) if np.sum(peak_mask_test) > 0 else 0
    mae_picos_avg = (mae_picos_train + mae_picos_test) / 2.0

    results_summary.append({
        'Split': split_name,
        'Best_Params': str(grid.best_params_),
        'MAE_Train': mae_train,
        'MAE_Test': mae_test,
        'MAE_Promedio': mae_avg,
        'RMSE_Train': rmse_train,
        'RMSE_Test': rmse_test,
        'R2_Train': r2_train,
        'R2_Test': r2_test,
        'MAE_Picos_Train': mae_picos_train,
        'MAE_Picos_Test': mae_picos_test,
        'MAE_Picos_Promedio': mae_picos_avg
    })

    if mae_avg < best_overall_mae:
        best_overall_mae = mae_avg
        best_split_name = split_name
        best_model = model
        best_split_data = {
            'y_train_real': y_train_real,
            'pred_train_real': pred_train_real,
            'y_test_real': y_test_real,
            'pred_test_real': pred_test_real,
            'mae_train': mae_train,
            'mae_test': mae_test,
            'mae_picos_train': mae_picos_train,
            'mae_picos_test': mae_picos_test,
            'dates_train': df['fecha'].iloc[:split_idx] if 'fecha' in df.columns else np.arange(split_idx),
            'dates_test': df['fecha'].iloc[split_idx:] if 'fecha' in df.columns else np.arange(split_idx, len(df))
        }

# Guardar DataFrame con reporte completo
df_results = pd.DataFrame(results_summary)
df_results.to_excel(os.path.join(PATH_RESULTS, "comparativa_splits_autoregresivo.xlsx"), index=False)

# -----------------------------------------------------------------------------
# 6. GENERACIÓN DE GRÁFICOS DE DESEMPEÑO
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharey=True)

# Entrenamiento
axes[0].plot(best_split_data['dates_train'], best_split_data['y_train_real'], label='Real (Entrenamiento)', color='blue', alpha=0.7)
axes[0].plot(best_split_data['dates_train'], best_split_data['pred_train_real'], label='Predicho (SVR)', color='darkorange', linestyle='--')
axes[0].set_title(f"Desempeño en Entrenamiento (Split {best_split_name}) - MAE: {best_split_data['mae_train']:.3f} | MAE Picos: {best_split_data['mae_picos_train']:.3f}", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Casos de Dengue")
axes[0].legend()
axes[0].grid(True, linestyle=':', alpha=0.6)

# Testeo
axes[1].plot(best_split_data['dates_test'], best_split_data['y_test_real'], label='Real (Testeo)', color='green', alpha=0.7)
axes[1].plot(best_split_data['dates_test'], best_split_data['pred_test_real'], label='Predicho (SVR)', color='red', linestyle='--')
axes[1].set_title(f"Desempeño en Testeo (Split {best_split_name}) - MAE: {best_split_data['mae_test']:.3f} | MAE Picos: {best_split_data['mae_picos_test']:.3f}", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Fecha")
axes[1].set_ylabel("Casos de Dengue")
axes[1].legend()
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plot_path = os.path.join(PATH_RESULTS, "comparacion_desempeno_autoregresivo.png")
plt.savefig(plot_path, dpi=300)
plt.close()

# -----------------------------------------------------------------------------
# 7. VERIFICACIÓN DE MAE < 4
# -----------------------------------------------------------------------------
print("=" * 60)
print("RESULTADOS CON PREDICTORES AUTOREGRESIVOS (REZAGOS DE CASOS_DENGUE)")
print("=" * 60)
print(f"\nMejor Split: {best_split_name}")
print(f"MAE Train: {best_split_data['mae_train']:.3f}")
print(f"MAE Test: {best_split_data['mae_test']:.3f}")
print(f"MAE Picos Train: {best_split_data['mae_picos_train']:.3f}")
print(f"MAE Picos Test: {best_split_data['mae_picos_test']:.3f}")
print(f"MAE Promedio: {best_overall_mae:.3f}")
print(f"Features seleccionados: {len(selected_features)}")

# Verificar condición MAE < 4
all_mae_below_4 = (
    best_split_data['mae_train'] < 4 and 
    best_split_data['mae_test'] < 4 and 
    best_split_data['mae_picos_train'] < 4 and 
    best_split_data['mae_picos_test'] < 4
)

if all_mae_below_4:
    print("\n✅ ¡ÉXITO! Todos los MAE son menores a 4.")
    print(f"   - MAE Train: {best_split_data['mae_train']:.3f} < 4 ✓")
    print(f"   - MAE Test: {best_split_data['mae_test']:.3f} < 4 ✓")
    print(f"   - MAE Picos Train: {best_split_data['mae_picos_train']:.3f} < 4 ✓")
    print(f"   - MAE Picos Test: {best_split_data['mae_picos_test']:.3f} < 4 ✓")
else:
    print("\n⚠️ Los MAE no cumplen todos con la condición < 4.")
    if best_split_data['mae_train'] >= 4:
        print(f"   - MAE Train: {best_split_data['mae_train']:.3f} >= 4 ✗")
    if best_split_data['mae_test'] >= 4:
        print(f"   - MAE Test: {best_split_data['mae_test']:.3f} >= 4 ✗")
    if best_split_data['mae_picos_train'] >= 4:
        print(f"   - MAE Picos Train: {best_split_data['mae_picos_train']:.3f} >= 4 ✗")
    if best_split_data['mae_picos_test'] >= 4:
        print(f"   - MAE Picos Test: {best_split_data['mae_picos_test']:.3f} >= 4 ✗")
    print("\nSugerencia: Ajustar top_k o incluir más rezagos para mejorar el rendimiento.")

print(f"\nArchivos guardados en: {PATH_RESULTS}")

RESULTADOS CON PREDICTORES AUTOREGRESIVOS (REZAGOS DE CASOS_DENGUE)

Mejor Split: 95-5
MAE Train: 5.610
MAE Test: 7.053
MAE Picos Train: 11.385
MAE Picos Test: 7.319
MAE Promedio: 6.331
Features seleccionados: 8

⚠️ Los MAE no cumplen todos con la condición < 4.
   - MAE Train: 5.610 >= 4 ✗
   - MAE Test: 7.053 >= 4 ✗
   - MAE Picos Train: 11.385 >= 4 ✗
   - MAE Picos Test: 7.319 >= 4 ✗

Sugerencia: Ajustar top_k o incluir más rezagos para mejorar el rendimiento.

Archivos guardados en: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\5_svm\3_resultados
